# Does the Volatility Risk Premium actually PAY — and when does it BLOW UP? (Colab, token-free)

The first study aimed at a *paycheck*, not a coin flip. The VRP = implied vol (VIX) runs structurally ABOVE
the realized vol that follows. Selling vol harvests that gap. Two questions, answered honestly:
1. **Does it pay?** VRP = VIX − forward-realized-vol. Mean, how often the seller wins, the distribution.
2. **When does it blow up, and when does it pay best?** Regime split by VIX level + 200-day trend, and a
   stylized short-variance equity curve so you SEE the grind-up and the craters (2008/2018/2020).

This is a RISK-PREMIUM harvest: it pays *because* it loses violently sometimes. The study's job is to size
the paycheck AND the tail. Run top-to-bottom; last cell exports one file.


## capture setup


In [ ]:
import builtins as _bi
REPORT_LINES=[]; _realprint=_bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
except Exception:
    _pip('pandas'); _pip('numpy'); import pandas as pd, numpy as np

def yf_close(sym, start='1990-01-01'):
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    df = yf.download(sym, start=start, progress=False, auto_adjust=True)
    if not len(df): return None
    c = df['Close']; c = c.iloc[:,0] if hasattr(c,'columns') else c
    return pd.Series(np.asarray(c).ravel(), index=df.index).dropna()

def stooq(sym):
    import urllib.request, io
    raw = urllib.request.urlopen(f'https://stooq.com/q/d/l/?s={sym}&i=d', timeout=30).read().decode()
    df = pd.read_csv(io.StringIO(raw), parse_dates=['Date']).set_index('Date').sort_index()
    return df['Close'].dropna()

def load(sym_yf, sym_stooq):
    try:
        s = yf_close(sym_yf)
        if s is not None and len(s)>500: return s
    except Exception as e: print(sym_yf,'yf failed:',e)
    return stooq(sym_stooq)

spx = load('^GSPC','^spx')
vix = load('^VIX','^vix')
df = pd.DataFrame({'spx':spx,'vix':vix}).dropna()
df['ret'] = df['spx'].pct_change()
print('aligned', len(df), 'days', df.index[0].date(), '->', df.index[-1].date())


## Q1 — does it PAY? VRP = VIX − the realized vol that actually followed


In [ ]:
H = 21   # ~1 month forward window
# forward realized vol (annualized %, over next H trading days)
fwd_rv = df['ret'].shift(-1).rolling(H).std().shift(-(H-1)) * np.sqrt(252) * 100
df['rv_fwd'] = fwd_rv
df['vrp'] = df['vix'] - df['rv_fwd']            # + = seller was overpaid (edge); - = realized beat implied
v = df['vrp'].dropna()
print(f'VRP = VIX minus subsequent {H}-day realized vol (both annualized, vol points):')
print(f'  mean {v.mean():+.2f} pts | median {v.median():+.2f} | seller WINS {100*(v>0).mean():.0f}% of days')
print(f'  best (implied way over) {v.max():+.1f} | worst (realized way over) {v.min():+.1f}')
print(f'  worst 5 days (the blow-ups):')
for dt,val in v.nsmallest(5).items(): print(f'    {dt.date()}  VRP {val:+.1f}  (VIX {df.loc[dt,"vix"]:.0f} vs realized {df.loc[dt,"rv_fwd"]:.0f})')
print('\nPositive mean + high win% = a real, persistent premium. The negative tail = why it pays.')


## Q2a — WHEN does it pay? VRP by VIX level and by 200-day regime


In [ ]:
d = df.dropna(subset=['vrp']).copy()
d['sma200'] = d['spx'].rolling(200).mean(); d['bull'] = d['spx']>d['sma200']
d['vixq'] = pd.qcut(d['vix'], 4, labels=['Q1 low VIX','Q2','Q3','Q4 high VIX'])
print('VRP by VIX quartile (is the premium bigger/safer when VIX is already elevated?):')
for q in ['Q1 low VIX','Q2','Q3','Q4 high VIX']:
    s = d.loc[d['vixq']==q,'vrp']
    print(f'  {q:12s}: mean VRP {s.mean():+5.2f} | win {100*(s>0).mean():3.0f}% | worst {s.min():+6.1f} (n={len(s)})')
print('\nVRP by 200-day regime:')
for nm,mask in [('BULL (px>200d)',d['bull']),('BEAR (px<200d)',~d['bull'])]:
    s = d.loc[mask,'vrp']
    print(f'  {nm:16s}: mean VRP {s.mean():+5.2f} | win {100*(s>0).mean():3.0f}% | worst {s.min():+6.1f} (n={len(s)})')


## Q2b — stylized SHORT-VARIANCE harvest: the grind-up and the craters


In [ ]:
# Each day: short variance struck at today's VIX, pay tomorrow's realized r^2 (constant notional, stylized).
# daily pnl (variance points, annualized-ish) = (VIX/100)^2/252  -  r_next^2   ... scaled to bp for readability
d = df.copy()
d['carry'] = (d['vix']/100.0)**2 / 252.0
d['realized'] = d['ret'].shift(-1)**2
d['pnl'] = (d['carry'] - d['realized']) * 1e4     # in 'variance bp'
d = d.dropna(subset=['pnl'])
def curve_stats(pnl, label):
    eq = pnl.cumsum(); dd = (eq - eq.cummax())
    sharpe = pnl.mean()/pnl.std()*np.sqrt(252) if pnl.std()>0 else float('nan')
    return (f'  {label:22s}: total {eq.iloc[-1]:8.0f} | Sharpe {sharpe:4.2f} | maxDD {dd.min():8.0f} '
            f'| worst day {pnl.min():7.0f} | up-days {100*(pnl>0).mean():3.0f}%')
print('Stylized short-variance harvest (constant notional):')
print(curve_stats(d['pnl'],'ALWAYS short vol'))
# regime-conditioned: only short when VIX elevated (>median), and only in bull
med = d['vix'].median()
print(curve_stats(d.loc[d['vix']>med,'pnl'],'only when VIX>median'))
d['sma200']=d['spx'].rolling(200).mean()
print(curve_stats(d.loc[d['spx']>d['sma200'],'pnl'].dropna(),'only in BULL (px>200d)'))
print(curve_stats(d.loc[(d['spx']>d['sma200'])&(d['vix']>med),'pnl'].dropna(),'BULL & VIX>median'))
print('\n% of ALL-short total given back in the single worst drawdown:')
eq=d['pnl'].cumsum(); dd=(eq-eq.cummax())
print(f'  maxDD {dd.min():.0f} vs total {eq.iloc[-1]:.0f}  -> gave back {100*abs(dd.min())/max(eq.iloc[-1],1):.0f}% of the whole run in one crash.')
print('Read: short-vol pays a high win-rate grind; the edge is REAL but the tail is the whole story -')
print('sizing + when-to-stand-down (regime) is the difference between harvest and ruin.')


### How to read / caveats
- **This is stylized** (constant-notional short variance, ignores option-specific greeks, roll, bid/ask, and
  that you'd never sell naked). It measures the PREMIUM and its RISK SHAPE, not a live P&L you can trade.
- The honest edge is in the *combination*: positive mean VRP + high win-rate (Q1) is the paycheck; the
  negative tail + max-drawdown (Q2) is the bill. **The regime splits are the actionable part** — if short-vol
  is safer/bigger in bull + elevated-VIX, that's *when* the premium is worth harvesting, and *when to stand
  down* is the vault's fragility signals ([[market-fragility]]) firing.
- **Goodhart/decay:** post-2018 'volmageddon' and the 2020 crash reshaped this trade. Rerun with
  `df = df[df.index>='2019-01-01']` to see the modern premium vs the fuller history.
- Paste the tables back and we'll write the verdict into [[where-the-edge-is]] and design a regime-gated
  premium-harvest around it.
